# もぐらたたき（p5.js ゲーム）

穴から顔を出すもぐらを、制限時間内にできるだけたくさんクリックするゲームです。

## ルール
- もぐらをクリックすると **1 点**。もぐらは短い時間で引っ込みます
- 制限時間は **30 秒**

## 操作
- **クリック**: もぐらをたたく / スタート / もう一度遊ぶ

## このノートブックの使い方

- コードセルを上から順番に **Shift + Enter** で実行し、最後の `%show` セルを実行するとゲーム画面が表示されます。
- キーボードで操作するゲームは、**最初にゲーム画面をクリック** してから操作してください（クリックでキー入力が画面に届くようになります）。
- コードを書き換えたら、そのセルを実行し直してから `%show` をもう一度実行すると、新しいゲームになります。
- 動かなくなったら、メニューの **Kernel → Restart Kernel and Clear Outputs of All Cells...** で最初からやり直せます。

p5.js の基本は `p5-tutorial.ipynb` で学べます。

## 1. ゲームの状態

時間の管理には `millis()`（スケッチ開始からのミリ秒）を使います。
「スタートした時刻」を覚えておけば、`millis() - startTime` で経過時間が分かります。

In [ ]:
const HOLES = [                      // 3 × 3 の穴の中心座標
  { x: 80, y: 120 }, { x: 200, y: 120 }, { x: 320, y: 120 },
  { x: 80, y: 230 }, { x: 200, y: 230 }, { x: 320, y: 230 },
  { x: 80, y: 340 }, { x: 200, y: 340 }, { x: 320, y: 340 },
];
const GAME_TIME = 30;                // 制限時間（秒）
const MOLE_R = 30;                   // もぐらの半径

let state = "ready";                 // "ready", "play", "finished"
let startTime = 0;                   // スタートした時刻（ミリ秒）
let score = 0;
let moleIndex = -1;                  // もぐらが出ている穴の番号（-1 = 出ていない）
let moleUntil = 0;                   // もぐらが引っ込む時刻
let nextMoleAt = 0;                  // 次にもぐらが出る時刻
let hitEffect = 0;                   // たたいた演出の残り時間

## 2. setup と draw

`draw()` では「時間の更新 → もぐらの出し入れ → 描画」を行います。
もぐらは `nextMoleAt` の時刻になったらランダムな穴に出て、`moleUntil` の時刻になったら引っ込みます。

In [ ]:
function setup() {
  createCanvas(400, 400);
  textFont("sans-serif");
}

function draw() {
  background(120, 190, 90);   // 草地

  let remaining = GAME_TIME;
  if (state === "play") {
    const now = millis();
    remaining = GAME_TIME - (now - startTime) / 1000;

    if (remaining <= 0) {
      remaining = 0;
      state = "finished";
      moleIndex = -1;
    } else {
      // もぐらを出す・引っ込める
      if (moleIndex === -1 && now >= nextMoleAt) {
        moleIndex = floor(random(HOLES.length));
        moleUntil = now + random(500, 1000);          // 0.5〜1 秒だけ出る
      } else if (moleIndex !== -1 && now >= moleUntil) {
        moleIndex = -1;
        nextMoleAt = now + random(200, 700);          // 少し休んでから次
      }
    }
  }

  // 穴ともぐら
  for (let i = 0; i < HOLES.length; i++) {
    const h = HOLES[i];
    noStroke();
    fill(70, 45, 20);
    ellipse(h.x, h.y + 15, 90, 40);                  // 穴
    if (i === moleIndex) {
      drawMole(h.x, h.y);
    }
  }

  // たたいた演出
  if (hitEffect > 0) {
    fill(255, 255, 0, hitEffect * 12);
    textSize(28);
    textAlign(CENTER, CENTER);
    text("ヒット!", mouseX, mouseY - 30);
    hitEffect--;
  }

  // 情報表示
  fill(255);
  stroke(0);
  strokeWeight(3);
  textSize(20);
  textAlign(LEFT, TOP);
  text("スコア: " + score, 10, 10);
  textAlign(RIGHT, TOP);
  text("残り: " + ceil(remaining) + " 秒", width - 10, 10);
  noStroke();

  if (state === "ready") {
    drawOverlay("もぐらたたき", "クリックでスタート");
  } else if (state === "finished") {
    drawOverlay("終了！ スコア: " + score, "クリックでもう一度");
  }
}

function drawMole(x, y) {
  noStroke();
  fill(150, 100, 60);
  circle(x, y, MOLE_R * 2);          // 顔
  fill(255);
  circle(x - 10, y - 6, 10);         // 目
  circle(x + 10, y - 6, 10);
  fill(0);
  circle(x - 10, y - 6, 4);
  circle(x + 10, y - 6, 4);
  fill(240, 150, 150);
  ellipse(x, y + 6, 12, 8);          // 鼻
}

function drawOverlay(title, sub) {
  fill(0, 160);
  rect(0, 0, width, height);
  fill(255);
  textAlign(CENTER, CENTER);
  textSize(32);
  text(title, width / 2, height / 2 - 20);
  textSize(18);
  text(sub, width / 2, height / 2 + 25);
}

## 3. 入力

クリックした位置ともぐらの中心の距離を `dist()` で測り、半径より近ければヒットです。

In [ ]:
function mousePressed() {
  if (state === "ready" || state === "finished") {
    startGame();
    return;
  }
  if (moleIndex !== -1) {
    const h = HOLES[moleIndex];
    if (dist(mouseX, mouseY, h.x, h.y) < MOLE_R) {
      score++;
      hitEffect = 20;
      moleIndex = -1;                            // すぐ引っ込める
      nextMoleAt = millis() + random(200, 600);
    }
  }
}

function startGame() {
  state = "play";
  score = 0;
  startTime = millis();
  moleIndex = -1;
  nextMoleAt = millis() + 500;
}

In [ ]:
%show 100% 410px

## 改造のヒント

- もぐらが出ている時間（`random(500, 1000)`）を短くすると難しくなります
- 同時に 2 匹出るようにしてみましょう（`moleIndex` を配列にする）
- たたいてはいけない「お邪魔キャラ」を追加してみましょう
- 残り時間が少なくなったら文字を赤くするなど、演出を工夫してみましょう